# Running Chiron on CPU (Windows + Docker) — Setup Guide

This document covers how to get [Chiron](https://github.com/haotianteng/Chiron) — a deep learning basecaller for Oxford Nanopore sequencing data — running on a **CPU-only Windows machine**, plus the specific fixes required to get the official repo working, since the vanilla `master` branch does not run out of the box.

## Summary of the problem

The official Chiron repo has three issues that block a clean CPU install today:

1. Its `Dockerfile.py3.cpu` pulls an unpinned `latest-py3` TensorFlow base image, which now resolves to TensorFlow 2.x — but Chiron's code uses TensorFlow 1.x-only APIs (`tensorflow.contrib`), so it crashes immediately.
2. `setup.py install` relies on an old dependency-fetching mechanism that fails silently on this old base image, and one dependency (`mappy`) needs a system library (`zlib`) that isn't present in the base image.
3. The `master` branch's code has diverged from the pretrained model checkpoint bundled in the repo (different internal variable names), causing a `NotFoundError` when restoring the model.
4. The bundled example `.fast5` test files are old enough that they're missing a `read_id` HDF5 attribute that the current extraction code expects, causing every file to silently fail to extract (with no visible error).

## Changes made to the original repo

### 1. `Dockerfile.py3.cpu` — pin TensorFlow version and fix dependency install

**Before:**

In [ ]:
# language: dockerfile
FROM tensorflow/tensorflow:latest-py3

WORKDIR /opt/chiron
COPY . .
RUN ["python", "setup.py", "install"]

WORKDIR /data

**After:**

In [ ]:
# language: dockerfile
FROM tensorflow/tensorflow:1.15.0-py3

WORKDIR /opt/chiron
COPY . .

RUN apt-get update && apt-get install -y zlib1g-dev

RUN pip install --upgrade pip
RUN pip install h5py mappy numpy statsmodels tqdm scipy biopython packaging
RUN python setup.py install

WORKDIR /data

**Why:**
- Pins the base image to TensorFlow **1.15.0**, matching what Chiron's code and `setup.py` actually expect (`tensorflow==1.15.0` is the pinned `extras_require` dependency).
- Installs `zlib1g-dev` at the OS level, required to compile the `mappy` Python dependency from source.
- Installs all Python dependencies explicitly via `pip` *before* running `setup.py install`, since letting `setup.py install` fetch them itself fails silently on this old image.

### 2. Checked out release tag `0.6.1` instead of `master`

In [ ]:
# language: bash
git fetch --tags
git checkout 0.6.1

**Why:** The `master` branch contains experimental changes (the maintainer confirms this in [Issue #80](https://github.com/haotianteng/Chiron/issues/80)) that change internal batch-normalization variable names, which no longer match the pretrained `DNA_default` checkpoint bundled in the repo. Checking out the last tagged release resolves the mismatch.

### 3. Patched `chiron/utils/extract_sig_ref.py` to tolerate a missing `read_id` attribute

**Before (around line 126):**

In [ ]:
# language: python
read_id = read_h.attrs['read_id'].decode('utf-8')

**After:**

In [ ]:
# language: python
try:
    read_id = read_h.attrs['read_id'].decode('utf-8')
except (KeyError, AttributeError):
    read_id = os.path.basename(input_file).split('.')[0]

**Why:** The bundled example `.fast5` files (in `chiron/example_data/DNA/`) predate the `read_id` HDF5 attribute this code expects. Without this fix, every file fails extraction silently (the exception is caught and logged to `output/log/extract.log`, not shown in the terminal), producing an empty output folder with no visible error.

---

## Prerequisites

- Windows 10 (build 19041+) or Windows 11
- No GPU required — this setup runs entirely on CPU (slower than GPU, but fully functional)

## Setup instructions

### 1. Install WSL2

Open **PowerShell as Administrator**:

In [ ]:
# language: powershell
wsl --install

Restart your computer when prompted. This installs WSL2 with Ubuntu by default (confirm with `wsl -l -v`, which should show version `2`).

### 2. Install Docker Desktop

1. Download from [docker.com/products/docker-desktop](https://www.docker.com/products/docker-desktop/)
2. Run the installer, keep **"Use WSL 2"** checked
3. Restart if prompted, then launch Docker Desktop
4. Verify:
   

In [ ]:
# language: powershell
   docker --version
   docker run hello-world
   

### 3. Install Git (if not already installed)

In [ ]:
# language: powershell
git --version

If missing, install from [git-scm.com/download/win](https://git-scm.com/download/win).

### 4. Clone this repo and check out the working release tag

In [ ]:
# language: powershell
git clone https://github.com/haotianteng/Chiron.git
cd Chiron
git fetch --tags
git checkout 0.6.1

### 5. Apply the two code fixes above

- Replace the contents of `Dockerfile.py3.cpu` with the fixed version shown above.
- Edit `chiron/utils/extract_sig_ref.py` to add the `try/except` fallback around the `read_id` line shown above.

### 6. Build the Docker image

In [ ]:
# language: powershell
docker build -t chiron-cpu -f Dockerfile.py3.cpu .

Takes a few minutes on first build.

### 7. Run the bundled example (5 test reads) to verify everything works

In [ ]:
# language: powershell
docker run -v ${PWD}:/data chiron-cpu chiron call -i /data/chiron/example_data/DNA -o /data/output -m /data/chiron/model/DNA_default --preset dna-pre

This should take under a minute on CPU (~40 seconds observed).

### 8. Check the output

In [ ]:
# language: powershell
dir output\result
Get-Content output\result\read1.fastq

You should see 5 `.fastq` files, each containing a basecalled DNA sequence (a `@` header line, an A/C/G/T sequence line, a `+` line, and a quality-score line).

---

## Running on your own data

In [ ]:
# language: powershell
docker run -v <path-to-your-fast5-folder>:/data chiron-cpu chiron call -i /data/<your-input-folder> -o /data/output -m /data/chiron/model/DNA_default --preset dna-pre

Add `-e fastq` explicitly if you want to force FASTQ output (with quality scores), or `-e fasta` for plain sequence output with no quality scores.

## Notes

- This setup runs on **CPU only** — per the original Chiron paper, expect roughly **21 bp/s** on CPU vs. **~1600+ bp/s** on GPU. Fine for small test sets; slow for large-scale basecalling.
- If you hit the same `read_id` extraction error on your *own* `.fast5` files, it likely means they also predate this attribute — the same patch (step 5) should resolve it.